# Caso Práctico 1: Motor de recomendación de películas

En este caso práctico vamos a construir tres sistemas de recomendación utilizando las valoraciones de MovieLens.

Primero veremos qué películas son las más populares. Después haremos filtrado colaborativo de dos formas: buscando usuarios con gustos parecidos y buscando películas que suelen recibir valoraciones similares. La idea es comparar los tres métodos y entender en qué situación tiene más sentido utilizar cada uno.


## 1. Imports, carga y exploración de los datos

Vamos a usar `pandas` para preparar los datos, `numpy` para calcular la similitud coseno y `matplotlib` y `plotly` para representar los resultados.

Al ejecutar la siguiente celda tenemos que seleccionar a la vez `file.tsv` y `Movie_Id_Titles.csv`.


In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from google.colab import files

# Subimos directamente los dos ficheros.
files.upload()

valoraciones = pd.read_csv(
    'file.tsv',
    sep='\t',
    names=['usuario_id', 'pelicula_id', 'rating', 'timestamp']
)
peliculas = pd.read_csv('Movie_Id_Titles.csv')

# Unimos valoraciones y títulos. La fecha no aporta nada a estos filtros.
datos = (valoraciones.drop(columns='timestamp')
         .merge(peliculas, left_on='pelicula_id', right_on='item_id', how='left')
         .drop(columns='item_id'))

print(f'Valoraciones: {len(datos):,}')
print(f'Usuarios: {datos.usuario_id.nunique():,}')
print(f'Películas: {datos.pelicula_id.nunique():,}')
print(f'Valores nulos: {datos.isna().sum().sum()}')
datos.head()


Saving file.tsv to file.tsv
Saving Movie_Id_Titles.csv to Movie_Id_Titles.csv
Valoraciones: 100,003
Usuarios: 944
Películas: 1,682
Valores nulos: 0


,usuario_id,pelicula_id,rating,title
0,0,50,5,Star Wars (1977)
1,0,172,5,"Empire Strikes Back, The (1980)"
2,0,133,1,Gone with the Wind (1939)
3,196,242,3,Kolya (1996)
4,186,302,3,L.A. Confidential (1997)


El dataset contiene el identificador del usuario, la película y una puntuación de 1 a 5. No tenemos información sobre género, edad o tipo de película, así que las recomendaciones dependerán únicamente de las valoraciones.

Una celda vacío no es una mala nota, simplemente no sabemos si ese usuario ha visto la película.

## 2. Recomendación basada en popularidad

Empezamos por el método más sencillo. Consideraremos más popular la película que haya recibido más valoraciones.

También mostraremos la nota media para tener algo de contexto, pero no la utilizaremos para ordenar. Una película con muchos votos no tiene por qué ser la mejor valorada; simplemente es la que más usuarios han puntuado.


In [5]:
# Calculamos popularidad y nota media.
popularidad = (datos.groupby(['pelicula_id', 'title']).rating
               .agg(num_valoraciones='count', nota_media='mean')
               .sort_values('num_valoraciones', ascending=False))

top_10_titulos = popularidad.head(10).index.get_level_values('title').tolist()
display(popularidad.head(10))

# En la gráfica situamos las 200 películas con más votos y destacamos el Top 10.
grafica_df = popularidad.head(200).reset_index()
grafica_df['grupo'] = grafica_df.title.map(
    lambda titulo: 'Top 10 popular' if titulo in top_10_titulos else 'Resto'
)
grafica_df['tamano'] = grafica_df.grupo.map({'Top 10 popular': 5.0, 'Resto': 2.0})

fig = px.scatter(
    grafica_df,
    x='num_valoraciones',
    y='nota_media',
    hover_name='title',
    color='nota_media',
    symbol='grupo',
    symbol_map={'Top 10 popular': 'star', 'Resto': 'circle'},
    size='tamano',
    title='Relación entre popularidad y nota media',
    labels={
        'num_valoraciones': 'Número de valoraciones',
        'nota_media': 'Nota media'
    },
    color_continuous_scale='RdYlGn'
)
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(height=600, template='plotly_white', showlegend=False)
fig.show()


,,num_valoraciones,nota_media
pelicula_id,title,,
50,Star Wars (1977),584,4.359589
258,Contact (1997),509,3.803536
100,Fargo (1996),508,4.155512
181,Return of the Jedi (1983),507,4.007890
294,Liar Liar (1997),485,3.156701
286,"English Patient, The (1996)",481,3.656965
288,Scream (1996),478,3.441423
1,Toy Story (1995),452,3.878319
300,Air Force One (1997),431,3.631090


Este método funciona bien para un usuario nuevo porque no necesita saber nada sobre él. El problema es que devuelve prácticamente la misma lista para todo el mundo y favorece siempre a las películas más conocidas.


## 3. Matriz usuario-película

Para los dos filtros colaborativos necesitamos convertir los datos en una matriz. Cada fila representa a un usuario, cada columna una película y cada celda contiene la valoración.

Vamos a mantener los valores vacíos como `NaN`. Rellenarlos con cero sería asumir que el usuario ha visto la película y le ha puesto la peor nota, y eso cambiaría completamente el significado de los datos.


In [3]:
matriz_usuario_pelicula = datos.pivot_table(
    index='usuario_id',
    columns='title',
    values='rating'
)

porcentaje_vacio = matriz_usuario_pelicula.isna().mean().mean()
print('Dimensiones:', matriz_usuario_pelicula.shape)
print(f'Porcentaje sin valorar: {porcentaje_vacio:.2%}')
matriz_usuario_pelicula.iloc[:5, :8]


Dimensiones: (944, 1664)
Porcentaje sin valorar: 93.65%


title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968)
usuario_id,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Filtrado colaborativo basado en usuarios

En este caso buscamos usuarios que hayan puntuado de forma parecida. Vamos a probar las dos medidas que propone el enunciado:

- **Pearson** se fija en el patrón de las notas. Puede detectar gustos parecidos aunque una persona sea más generosa puntuando que otra.
- **Coseno** comprueba si las valoraciones apuntan en una dirección parecida. No corrige tanto la forma de puntuar, pero funciona bien cuando la matriz tiene muchos huecos.

No debemos comparar sus números como si fueran exactamente la misma escala. Lo que nos interesa es ver que ambas permiten ordenar usuarios parecidos. En los dos casos exigimos un mínimo de 20 películas en común para no sacar conclusiones a partir de una coincidencia demasiado pequeña.


In [4]:
def calcular_similitudes(matriz, objetivo, metodo='coseno', minimo_comun=20):
    # Primero dejamos solo las filas con suficientes valoraciones en común.
    coincidencias = matriz.notna().mul(objetivo.notna(), axis='columns').sum(axis=1)
    candidatas = matriz.loc[coincidencias >= minimo_comun]

    if metodo == 'pearson':
        # Algunas parejas no tienen variación suficiente. En ese caso Pearson
        # devuelve NaN, que descartaremos después, sin llenar la salida de avisos.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            return candidatas.corrwith(
                objetivo,
                axis=1,
                min_periods=minimo_comun
            )

    if metodo == 'coseno':
        # Para cada fila usamos únicamente las posiciones valoradas por ambos.
        valores = candidatas.fillna(0).mul(objetivo.notna(), axis='columns')
        objetivo_comun = candidatas.notna().mul(
            objetivo.fillna(0),
            axis='columns'
        )
        numerador = (valores * objetivo_comun).sum(axis=1)
        denominador = np.sqrt(
            (valores ** 2).sum(axis=1) *
            (objetivo_comun ** 2).sum(axis=1)
        )
        return numerador / denominador.replace(0, np.nan)

    raise ValueError("El método debe ser 'pearson' o 'coseno'.")


USUARIO_EJEMPLO = 196
objetivo_ejemplo = matriz_usuario_pelicula.loc[USUARIO_EJEMPLO]

similitudes_pearson = calcular_similitudes(
    matriz_usuario_pelicula, objetivo_ejemplo, metodo='pearson', minimo_comun=20
).drop(index=USUARIO_EJEMPLO, errors='ignore').dropna()

similitudes_coseno = calcular_similitudes(
    matriz_usuario_pelicula, objetivo_ejemplo, metodo='coseno', minimo_comun=20
).drop(index=USUARIO_EJEMPLO, errors='ignore').dropna()

resumen_metricas = pd.DataFrame({
    'Métrica': ['Pearson', 'Coseno'],
    'Usuarios comparables': [len(similitudes_pearson), len(similitudes_coseno)],
    'Similitudes positivas': [
        (similitudes_pearson > 0).sum(),
        (similitudes_coseno > 0).sum()
    ],
    'Mayor similitud': [similitudes_pearson.max(), similitudes_coseno.max()]
})

display(resumen_metricas.round(3))


TypeError: DataFrame.corrwith() got an unexpected keyword argument 'min_periods'

Para generar las recomendaciones nos quedaremos con el coseno. En este dataset hay muchísimas películas sin valorar y esta medida se adapta bien a ese tipo de matriz. También es la que utiliza CineMatch cuando recibe un perfil corto desde la web. Pearson sigue siendo útil para la comparación, pero aquí preferimos mantener el mismo criterio en el notebook y en la demostración.


In [ ]:
def recomendar_por_usuarios(
    usuario_id,
    top_n=10,
    minimo_comun=20,
    metodo='coseno'
):
    objetivo = matriz_usuario_pelicula.loc[usuario_id]

    # Calculamos la similitud y eliminamos al propio usuario.
    similitudes = (calcular_similitudes(
        matriz_usuario_pelicula,
        objetivo,
        metodo=metodo,
        minimo_comun=minimo_comun
    ).drop(index=usuario_id, errors='ignore').dropna())
    vecinos = similitudes[similitudes > 0].sort_values(ascending=False).head(20)

    predicciones = []
    for titulo in objetivo[objetivo.isna()].index:
        notas = matriz_usuario_pelicula.loc[vecinos.index, titulo].dropna()
        pesos = vecinos.loc[notas.index]
        if len(notas) >= 2 and pesos.sum() > 0:
            predicciones.append({
                'title': titulo,
                'prediccion': (notas * pesos).sum() / pesos.sum(),
                'vecinos_que_la_valoran': len(notas)
            })

    return (pd.DataFrame(predicciones)
            .sort_values(['prediccion', 'vecinos_que_la_valoran'], ascending=False)
            .head(top_n)), vecinos

recomendaciones_usuario, vecinos = recomendar_por_usuarios(
    USUARIO_EJEMPLO,
    metodo='coseno'
)

print('Métrica utilizada: coseno')
print(f'Vecinos utilizados: {len(vecinos)}')
display(recomendaciones_usuario)

mostrar = recomendaciones_usuario.sort_values('prediccion')
plt.figure(figsize=(10, 6))
plt.barh(mostrar.title, mostrar.prediccion, color='#D07A93')
plt.title(f'Recomendaciones para el usuario {USUARIO_EJEMPLO} basadas en usuarios similares')
plt.xlabel('Valoración estimada')
plt.xlim(0, 5)
plt.tight_layout()
plt.show()


Ahora sí obtenemos una lista personalizada. Dos usuarios pueden recibir resultados diferentes porque sus vecinos también serán diferentes.

La principal limitación es que necesitamos suficientes películas en común para calcular una similitud fiable. Si el usuario acaba de llegar o tiene gustos muy poco habituales, puede que no encontremos vecinos útiles.


## 5. Filtrado colaborativo basado en ítems

El tercer método compara películas en lugar de personas. La lógica es sencilla: si dos películas suelen recibir notas parecidas de los mismos usuarios, consideramos que están relacionadas.

Usaremos también similitud coseno y exigimos al menos 50 valoraciones comunes. Para construir la lista personalizada partimos de las películas que el usuario ha puntuado con 4 o 5 y buscamos títulos similares. Personalmente, este método me parece más fácil de explicar porque podemos decir directamente qué película del historial ha provocado cada recomendación.


In [ ]:
matriz_pelicula_usuario = matriz_usuario_pelicula.T

def peliculas_similares(titulo, top_n=100, minimo_comun=50, metodo='coseno'):
    objetivo = matriz_pelicula_usuario.loc[titulo]
    similitudes = (calcular_similitudes(
        matriz_pelicula_usuario,
        objetivo,
        metodo=metodo,
        minimo_comun=minimo_comun
    ).drop(index=titulo, errors='ignore').dropna())
    return similitudes[similitudes > 0].sort_values(ascending=False).head(top_n)

def recomendar_por_items(usuario_id, top_n=10):
    historial = matriz_usuario_pelicula.loc[usuario_id].dropna()
    favoritas = historial[historial >= 4]
    candidatos = {}

    # Cada película favorita aporta candidatos según su similitud.
    for titulo, rating in favoritas.items():
        for candidata, similitud in peliculas_similares(titulo, metodo='coseno').items():
            if candidata in historial.index:
                continue
            fila = candidatos.setdefault(
                candidata,
                {
                    'afinidad': 0,
                    'similitud_principal': 0,
                    'porque': titulo,
                    'valoracion_origen': rating,
                    'conexiones': 0
                }
            )
            fila['conexiones'] += 1

            # La afinidad combina similitud y cuánto gustó la película de origen.
            afinidad = similitud * (rating / 5)
            if afinidad > fila['afinidad']:
                fila['afinidad'] = afinidad
                fila['similitud_principal'] = similitud
                fila['porque'] = titulo
                fila['valoracion_origen'] = rating

    recomendaciones = [
        {
            'title': titulo,
            'afinidad': fila['afinidad'],
            'similitud_principal': fila['similitud_principal'],
            'porque': fila['porque'],
            'valoracion_origen': fila['valoracion_origen'],
            'conexiones': fila['conexiones']
        }
        for titulo, fila in candidatos.items()
    ]
    return (pd.DataFrame(recomendaciones)
            .sort_values(['afinidad', 'conexiones'], ascending=False)
            .head(top_n))

recomendaciones_items = recomendar_por_items(USUARIO_EJEMPLO)
display(recomendaciones_items)

mostrar = recomendaciones_items.sort_values('conexiones')
plt.figure(figsize=(10, 6))
barras = plt.barh(mostrar.title, mostrar.conexiones, color='#856084')
plt.title(f'Películas del historial que apoyan cada recomendación')
plt.xlabel('Número de conexiones con películas valoradas con 4 o 5')

for barra, similitud in zip(barras, mostrar.similitud_principal):
    plt.text(
        barra.get_width() + 0.15,
        barra.get_y() + barra.get_height() / 2,
        f'sim. {similitud:.3f}',
        va='center'
    )

plt.xlim(0, mostrar.conexiones.max() + 4)
plt.tight_layout()
plt.show()


La `afinidad` ya no intenta aparentar que es una nota esperada. Es un valor interno entre 0 y 1 que combina la similitud con la valoración de la película de origen. Cuanto más alto sea, más fuerte es esa relación, pero no es una probabilidad.

La columna `porque` permite ver qué película ha provocado principalmente la sugerencia y `conexiones` cuenta cuántas películas favoritas también la respaldan. Por eso la gráfica representa las conexiones y anota la similitud principal: aporta más información que diez barras prácticamente iguales a 5.

Este enfoque suele ser más estable que comparar usuarios, pero sigue dependiendo de que existan suficientes valoraciones comunes entre las películas. Los títulos menos conocidos van a tener más dificultades para aparecer.


## 6. Comparación de los resultados

No existe un método que sea siempre mejor. Cada uno resuelve una parte distinta del problema, así que vamos a resumir qué hemos obtenido y dónde falla cada enfoque.


In [ ]:
comparacion = pd.DataFrame({
    'Método': ['Popularidad', 'Usuarios similares', 'Ítems similares'],
    '¿Personaliza?': ['No', 'Sí', 'Sí'],
    'Necesita historial': ['No', 'Sí, y coincidencias con otros usuarios', 'Sí, y películas con suficientes votos'],
    'Punto fuerte': [
        'Funciona desde la primera visita',
        'Aprovecha gustos de personas parecidas',
        'Es estable y permite explicar el motivo'
    ],
    'Limitación principal': [
        'Recomienda lo mismo a todo el mundo',
        'Sufre cuando hay pocas valoraciones comunes',
        'Favorece películas con bastante historial'
    ]
})

comparacion


## 7. Conclusiones

En este caso práctico hemos implementado los tres métodos que pedía el enunciado utilizando `pandas`. También hemos comparado Pearson y coseno: las dos medidas sirven para encontrar patrones parecidos, pero hemos utilizado coseno en las recomendaciones porque se adapta bien a una matriz con muchos huecos y mantiene el mismo criterio que la web. El ranking de popularidad es el más sencillo y funciona bien para arrancar, aunque no tiene ninguna personalización. El filtrado por usuarios sí adapta el resultado, pero necesita suficientes coincidencias para encontrar vecinos fiables.

El filtrado basado en ítems es el que más me convence para este dataset. Además de personalizar, permite justificar cada resultado a partir de una película que el usuario ya ha valorado bien. Aun así, también tiene sesgo hacia los títulos con más información y no resuelve por sí solo el problema de un usuario completamente nuevo.

Como ampliación he desarrollado CineMatch, una web que representa visualmente estos tres enfoques. La aplicación añade una mezcla híbrida con pesos dinámicos: al principio se apoya más en popularidad y, cuando el perfil acumula valoraciones, da más peso a usuarios e ítems similares. También incorpora perfiles locales y metadatos de TMDB. Estas mejoras no sustituyen los tres filtros del ejercicio; simplemente los combinan y los presentan de una forma más cercana a una aplicación real.
